# Chapter 21: Model Deployment and Serving
**Part VI — MLOps & Production AI**

*The Complete MSc Reference: Artificial Intelligence & Machine Learning*  
*Jijesh Puliyappottammal — Digichange Publication, 2026*

---

FastAPI, Docker, cloud deployment, batch vs real-time inference, and canary deployments.

## Learning Objectives
See Chapter 21 in the main textbook for full learning objectives, theory, and exam guidance.

## How to Use These Notebooks
Run cells from top to bottom. All cells are self-contained. Install any missing packages with `pip install <package>` in a new cell.


In [ ]:
# Standard imports used throughout the book
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

# Verify Python and key package versions
import sys
print("Python:", sys.version.split()[0])
try:
    import numpy, sklearn, torch
    print("NumPy:", numpy.__version__)
    print("Scikit-learn:", sklearn.__version__)
    print("PyTorch:", torch.__version__)
except ImportError as e:
    print(f"Missing: {e.name} — run: pip install {e.name}")


## Dockerfile for the prediction API


In [ ]:
# Dockerfile for the prediction API
FROM python:3.11-slim

WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY model.pkl scaler.pkl app.py .

EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]

# Build and run
# $ docker build -t ml-api:v1.0 .
# $ docker run -p 8000:8000 ml-api:v1.0

##     # Log hyperparameters


In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

mlflow.set_experiment("fraud-detection-v2")

with mlflow.start_run(run_name="rf-100-trees"):
    # Log hyperparameters
    n_est = 100
    mlflow.log_param("n_estimators", n_est)
    mlflow.log_param("max_depth", 10)

    # Train
    clf = RandomForestClassifier(n_estimators=n_est, max_depth=10)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    # Log metrics
    f1 = f1_score(y_test, y_pred, average="macro")
    mlflow.log_metric("f1_macro", f1)

    # Log model
    mlflow.sklearn.log_model(clf, "model")
    print(f"Run complete. F1: {f1:.4f}")

## Model serialisation and loading — scikit-learn and PyTorch


In [ ]:
# Model serialisation and loading — scikit-learn and PyTorch
import joblib
import torch
import torch.nn as nn
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
import os

print("=" * 50)
print("Model Serialisation: scikit-learn (joblib)")
print("=" * 50)

X, y = make_classification(n_samples=1000, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clf = RandomForestClassifier(n_estimators=50, random_state=42).fit(X_train, y_train)
print(f"Test accuracy before save: {clf.score(X_test, y_test):.4f}")

joblib.dump(clf, '/tmp/rf_model.pkl')
print(f"Model saved: /tmp/rf_model.pkl ({os.path.getsize('/tmp/rf_model.pkl') / 1024:.1f} KB)")

loaded_clf = joblib.load('/tmp/rf_model.pkl')
print(f"Test accuracy after load:  {loaded_clf.score(X_test, y_test):.4f}")
print("✅ Accuracy preserved across serialisation")

print()
print("=" * 50)
print("Model Serialisation: PyTorch state_dict")
print("=" * 50)

class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(nn.Linear(20,64), nn.ReLU(), nn.Linear(64,2))
    def forward(self, x): return self.layers(x)

model = SimpleNet()
optimizer = torch.optim.Adam(model.parameters())

# Save full training checkpoint
torch.save({
    'epoch': 10,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': 0.1234,
}, '/tmp/checkpoint.pt')
print(f"Checkpoint saved: /tmp/checkpoint.pt ({os.path.getsize('/tmp/checkpoint.pt') / 1024:.1f} KB)")

# Reload
model2 = SimpleNet()
ckpt = torch.load('/tmp/checkpoint.pt', weights_only=True)
model2.load_state_dict(ckpt['model_state_dict'])
print(f"Loaded from epoch {ckpt['epoch']}, loss was {ckpt['loss']:.4f}")
print("✅ State dict approach is safer than pickle for production")

print()
print("Best practices for production model saving:")
print("  1. Always save: model weights, scaler/preprocessor, feature names, version")
print("  2. Use joblib for sklearn; state_dict for PyTorch (not pickle)")
print("  3. Version your models: model_v1.0.0.pkl, model_v1.1.0.pkl")
print("  4. Store model card alongside: training data version, metrics, date")

---

## 📚 Review Questions

See Chapter 21 of the textbook for:
- 5 review questions
- Common exam question with model answer (Appendix C)
- Flashcard summary (Appendix A)
- Formula sheet (Appendix B)

## 📖 Further Reading

See the Further Reading section at the end of Chapter 21 in the textbook.

---
*© 2026 Jijesh Puliyappottammal — Digichange Publication. Code examples are released under the MIT Licence for educational use.*